In [1]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path(
    r"C:\Thinh_Project\credit-card-churn-retention-analytics"
    r"\data\raw\credit_card_customer_churn_dataset.csv"
)

df = pd.read_csv(DATA_PATH)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
df.head()

Rows: 100,000
Columns: 40


,customer_id,age,gender,marital_status,education_level,employment_status,income,city,credit_score,account_age_months,...,support_satisfaction,reward_points,offers_received,offers_redeemed,financial_stress_score,engagement_score,customer_value_score,churn_risk_score,churn_probability,churned
0,CUST-0000001,50,Male,Divorced,Bachelor,Employed,31679.55,San Diego,685,151,...,4.1,728,3,1,50.33,74.89,54.35,13.00,0.1300,0
1,CUST-0000002,42,Female,Married,Bachelor,Self-Employed,83618.66,Los Angeles,732,81,...,3.3,13537,3,2,17.70,85.36,99.34,4.30,0.0430,0
2,CUST-0000003,52,Male,Married,Bachelor,Self-Employed,134295.30,Phoenix,777,41,...,3.2,4671,4,2,15.67,74.81,30.56,3.32,0.0332,0
3,CUST-0000004,63,Female,Married,Bachelor,Employed,100983.46,New York,708,54,...,3.1,12694,4,0,13.13,47.55,80.07,19.30,0.1930,0
4,CUST-0000005,40,Male,Married,Bachelor,Employed,89736.89,Other,664,95,...,4.5,37007,1,1,44.25,58.71,100.00,15.10,0.1510,0


In [2]:
data_dictionary = pd.DataFrame({
    "data_type": df.dtypes,
    "missing_count": df.isna().sum(),
    "unique_values": df.nunique(),
})

data_dictionary.index.name = "column_name"
data_dictionary

,data_type,missing_count,unique_values
column_name,,,
customer_id,str,0,100000
age,int64,0,68
gender,str,0,3
marital_status,str,0,4
education_level,str,0,5
employment_status,str,0,5
income,float64,0,98671
city,str,0,10
credit_score,int64,0,449


In [9]:
df.describe(include="all").round(2).T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
column_name,,,,,,,,,,,
customer_id,100000,100000,CUST-0000001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,100000.0,NaN,NaN,NaN,43.63,12.72,18.0,35.0,44.0,52.0,85.0
gender,100000,3,Female,49193,NaN,NaN,NaN,NaN,NaN,NaN,NaN
marital_status,100000,4,Married,48091,NaN,NaN,NaN,NaN,NaN,NaN,NaN
education_level,100000,5,Bachelor,39937,NaN,NaN,NaN,NaN,NaN,NaN,NaN
employment_status,100000,5,Employed,62082,NaN,NaN,NaN,NaN,NaN,NaN,NaN
income,100000.0,NaN,NaN,NaN,80669.83,52343.67,15000.0,44670.95,67196.33,101188.72,350000.0
city,100000,10,Other,42047,NaN,NaN,NaN,NaN,NaN,NaN,NaN
credit_score,100000.0,NaN,NaN,NaN,689.08,78.32,344.0,636.0,690.0,744.0,850.0


In [3]:
# Check duplicate records and customer IDs
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate customer IDs:", df["customer_id"].duplicated().sum())

# Summarize the churn label
churn_summary = (
    df["churned"]
    .value_counts(dropna=False)
    .rename_axis("churned")
    .to_frame("customer_count")
)

churn_summary["percentage"] = (
    churn_summary["customer_count"] / len(df) * 100
).round(2)

churn_summary

Duplicate rows: 0
Duplicate customer IDs: 0


,customer_count,percentage
churned,,
0,80542,80.54
1,19458,19.46


In [4]:
quality_checks = pd.Series({
    "Invalid churn labels": (
        ~df["churned"].isin([0, 1])
    ).sum(),

    "Online transactions exceed total": (
        df["online_transactions"] > df["monthly_transactions"]
    ).sum(),

    "International transactions exceed total": (
        df["international_transactions"] > df["monthly_transactions"]
    ).sum(),

    "Offers redeemed exceed received": (
        df["offers_redeemed"] > df["offers_received"]
    ).sum(),

    "Utilization outside 0–1": (
        ~df["utilization_ratio"].between(0, 1)
    ).sum(),

    "Satisfaction outside 1–5": (
        ~df["support_satisfaction"].between(1, 5)
    ).sum(),

    "Negative monthly spending": (
        df["monthly_spending"] < 0
    ).sum(),
}, name="flagged_rows")

quality_checks.to_frame()

,flagged_rows
Invalid churn labels,0
Online transactions exceed total,0
International transactions exceed total,0
Offers redeemed exceed received,0
Utilization outside 0–1,0
Satisfaction outside 1–5,0
Negative monthly spending,0


In [5]:
column_roles = pd.DataFrame({
    "column_name": df.columns,
    "role": [
        "identifier" if col == "customer_id"
        else "target" if col == "churned"
        else "excluded_pending_review" if col in [
            "churn_probability",
            "churn_risk_score",
            "financial_stress_score",
            "engagement_score",
            "customer_value_score",
        ]
        else "candidate_feature"
        for col in df.columns
    ]
})

column_roles

,column_name,role
0,customer_id,identifier
1,age,candidate_feature
2,gender,candidate_feature
3,marital_status,candidate_feature
4,education_level,candidate_feature
5,employment_status,candidate_feature
6,income,candidate_feature
7,city,candidate_feature
8,credit_score,candidate_feature
9,account_age_months,candidate_feature


## Data Understanding & Audit Summary

- **Dataset:** 100,000 customer records and 40 columns.
- **Quality:** No missing values, duplicate rows, or duplicate customer IDs detected. All seven initial validation checks passed.
- **Target:** `churned`; 19,458 customers (19.46%) are labeled as churned.
- **Column roles:** `customer_id` is an identifier. Five existing score/probability columns are excluded from model inputs pending review; remaining columns are candidate features.
- **Limitations:** Source, churn definition, observation period, and feature timing remain unverified. No date columns are available.

**Conclusion:** Ready for descriptive EDA using SQL and DuckDB. Findings describe associations; 19.46% is the dataset churn proportion, not a monthly churn rate.